In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Transpiler passes AI
Transpiler passes berkuasa AI adalah laluan yang berfungsi sebagai pengganti langsung laluan Qiskit "tradisional" untuk beberapa tugas transpilasi. Ia sering menghasilkan keputusan yang lebih baik daripada algoritma heuristik sedia ada (seperti kedalaman dan bilangan CNOT yang lebih rendah), tetapi juga jauh lebih pantas daripada algoritma pengoptimuman seperti penyelesai kepuasan Boolean. Transpiler passes AI boleh dijalankan di persekitaran tempatan anda atau di awan menggunakan Perkhidmatan Transpiler Qiskit jika anda adalah sebahagian daripada Plan Premium IBM Quantum&reg;, Plan Flex, atau Plan On-Prem (melalui IBM Quantum Platform API).

> **Note:** Transpiler passes berkuasa AI berada dalam status keluaran beta, tertakluk kepada perubahan.
>     Jika anda mempunyai maklum balas atau ingin menghubungi pasukan pembangun, sila gunakan [saluran Ruang Kerja Qiskit Slack](https://qiskit.slack.com/archives/C06KF8YHUAU) ini.

Laluan berikut tersedia pada masa ini:

**Laluan penghalaan**
 - `AIRouting`: Pemilihan reka letak dan penghalaan Circuit

**Laluan sintesis Circuit**
 - `AICliffordSynthesis`: Sintesis Circuit Clifford
 - `AILinearFunctionSynthesis`: Sintesis Circuit fungsi linear
 - `AIPermutationSynthesis`: Sintesis Circuit permutasi
 - `AIPauliNetworkSynthesis`: Sintesis Circuit Rangkaian Pauli

Untuk menggunakan transpiler passes AI, mula-mula [pasang pakej `qiskit-ibm-transpiler`](/guides/qiskit-transpiler-service#install-transpiler-service). Lawati [dokumentasi API qiskit-ibm-transpiler](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler) untuk mendapatkan lebih banyak maklumat tentang pilihan yang tersedia.

## Jalankan transpiler passes AI secara tempatan atau di awan
Jika anda ingin menggunakan transpiler passes berkuasa AI di persekitaran tempatan anda secara percuma, pasang `qiskit-ibm-transpiler` dengan beberapa kebergantungan tambahan seperti berikut:

In [1]:
from qiskit.transpiler import PassManager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_runtime import QiskitRuntimeService

backend = QiskitRuntimeService().backend("ibm_torino")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=backend,
            optimization_level=2,
            layout_mode="optimize",
            local_mode=True,
        )
    ]
)


circuit = efficient_su2(101, entanglement="circular", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

Tanpa kebergantungan tambahan ini, transpiler passes berkuasa AI dijalankan di awan melalui Perkhidmatan Transpiler Qiskit (hanya tersedia untuk pengguna Plan Premium IBM Quantum, Plan Flex, atau Plan On-Prem (melalui IBM Quantum Platform API)). Selepas memasang kebergantungan tambahan, mod lalai untuk menjalankan transpiler passes berkuasa AI adalah menggunakan mesin tempatan anda.

## Laluan penghalaan AI
Laluan `AIRouting` bertindak sebagai peringkat reka letak dan peringkat penghalaan. Ia boleh digunakan dalam `PassManager` seperti berikut:

In [ ]:
from qiskit.transpiler import PassManager

from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_transpiler.ai.synthesis import AILinearFunctionSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectLinearFunctions
from qiskit_ibm_transpiler.ai.synthesis import AIPauliNetworkSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectPauliNetworks
from qiskit.circuit.library import efficient_su2

ibm_torino = QiskitRuntimeService().backend("ibm_torino")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=ibm_torino,
            optimization_level=3,
            layout_mode="optimize",
            local_mode=True,
        ),  # Route circuit
        CollectLinearFunctions(),  # Collect Linear Function blocks
        AILinearFunctionSynthesis(
            backend=ibm_torino, local_mode=True
        ),  # Re-synthesize Linear Function blocks
        CollectPauliNetworks(),  # Collect Pauli Networks blocks
        AIPauliNetworkSynthesis(
            backend=ibm_torino, local_mode=True
        ),  # Re-synthesize Pauli Network blocks.
    ]
)

circuit = efficient_su2(10, entanglement="full", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

Di sini, `backend` menentukan peta gandingan yang hendak dihalakan, `optimization_level` (1, 2, atau 3) menentukan usaha pengiraan yang digunakan dalam proses (lebih tinggi biasanya memberikan keputusan lebih baik tetapi mengambil masa lebih lama), dan `layout_mode` menentukan cara mengendalikan pemilihan reka letak.
`layout_mode` termasuk pilihan berikut:

- `keep`: Ini menghormati reka letak yang ditetapkan oleh transpiler passes sebelumnya (atau menggunakan reka letak trivial jika tidak ditetapkan). Ia biasanya hanya digunakan apabila litar mesti dijalankan pada Qubit tertentu peranti. Ia sering menghasilkan keputusan yang lebih buruk kerana mempunyai lebih sedikit ruang untuk pengoptimuman.
- `improve`: Ini menggunakan reka letak yang ditetapkan oleh transpiler passes sebelumnya sebagai titik permulaan. Ia berguna apabila anda mempunyai tekaan awal yang baik untuk reka letak; contohnya, untuk litar yang dibina dengan cara yang kira-kira mengikuti peta gandingan peranti. Ia juga berguna jika anda ingin mencuba laluan reka letak tertentu lain yang digabungkan dengan laluan `AIRouting`.
- `optimize`: Ini adalah mod lalai. Ia berfungsi terbaik untuk litar umum di mana anda mungkin tidak mempunyai tekaan reka letak yang baik. Mod ini mengabaikan pemilihan reka letak sebelumnya.
- `local_mode`: Bendera ini menentukan di mana laluan `AIRouting` dijalankan. Jika `False`, `AIRouting` dijalankan dari jauh melalui Perkhidmatan Transpiler Qiskit. Jika `True`, pakej cuba menjalankan laluan di persekitaran tempatan anda dengan sandaran ke mod awan jika kebergantungan yang diperlukan tidak ditemui.

## Laluan sintesis Circuit AI
Laluan sintesis Circuit AI membolehkan anda mengoptimumkan bahagian jenis litar yang berbeza ([Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford), [Fungsi Linear](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction), [Permutasi](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation), Rangkaian Pauli) dengan mensintesiskan semula. Cara biasa untuk menggunakan laluan sintesis adalah seperti berikut:

In [ ]:
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import QiskitRuntimeService


backend = QiskitRuntimeService().backend("ibm_torino")
torino_coupling_map = backend.coupling_map


su2_circuit = efficient_su2(101, entanglement="circular", reps=1)

ai_transpiler_pass_manager = generate_ai_pass_manager(
    coupling_map=torino_coupling_map,
    ai_optimization_level=3,
    optimization_level=3,
    ai_layout_mode="optimize",
)

ai_su2_transpiled_circuit = ai_transpiler_pass_manager.run(su2_circuit)

Sintesis menghormati peta gandingan peranti: ia boleh dijalankan dengan selamat selepas laluan penghalaan lain tanpa mengganggu litar, jadi keseluruhan litar masih akan mengikuti sekatan peranti. Secara lalai, sintesis akan menggantikan sublitar asal hanya jika sublitar yang disintesis meningkatkan yang asal (pada masa ini hanya memeriksa bilangan CNOT), tetapi ini boleh dipaksa untuk sentiasa menggantikan litar dengan menetapkan `replace_only_if_better=False`.

Laluan sintesis berikut tersedia dari `qiskit_ibm_transpiler.ai.synthesis`:

- *AICliffordSynthesis*: Sintesis untuk litar [Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford) (blok get `H`, `S`, dan `CX`). Pada masa ini sehingga sembilan blok Qubit.
- *AILinearFunctionSynthesis*: Sintesis untuk litar [Fungsi Linear](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction) (blok get `CX` dan `SWAP`). Pada masa ini sehingga sembilan blok Qubit.
- *AIPermutationSynthesis*: Sintesis untuk litar [Permutasi](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation) (blok get `SWAP`). Pada masa ini tersedia untuk blok 65, 33, dan 27 Qubit.
- *AIPauliNetworkSynthesis*: Sintesis untuk litar Rangkaian Pauli (blok get `H`, `S`, `SX`, `CX`, `RX`, `RY` dan `RZ`). Pada masa ini sehingga enam blok Qubit.

Kami dijangka akan meningkatkan saiz blok yang disokong secara beransur-ansur.

Semua laluan menggunakan kumpulan benang untuk menghantar beberapa permintaan secara selari. Secara lalai, bilangan benang maksimum adalah bilangan teras tambah empat (nilai lalai untuk objek Python `ThreadPoolExecutor`). Walau bagaimanapun, anda boleh menetapkan nilai anda sendiri dengan argumen `max_threads` semasa instantiasi laluan. Contohnya, baris berikut menginstansiasikan laluan `AILinearFunctionSynthesis`, yang membolehkannya menggunakan maksimum 20 benang.